# LiH chemistry test: is the W* verdict flip reachable for a real molecule?

Honest, pre-committed test. We take **LiH/STO-3G (n=12)** with a standard **UCCSD** ansatz and ask
whether there is an operating point where the *same* physical circuit is Pauli-propagation-HARD under
Jordan-Wigner but classically simulable under Bravyi-Kitaev (a verdict flip), with the physical
expectation value identical across encodings. The threshold is fixed at 2 log2 n; the full
operating-point x seed sweep is reported, so a flip (or its absence) is visible without cherry-picking.
Self-validating: the engine is checked against independent ground truth before the test.


In [ ]:
print(">>> LiH chemistry flip test <<<")
!pip install -q openfermion joblib

In [ ]:
# ============================ validated engine (inlined verbatim) ============================
import numpy as np
import openfermion as of
from openfermion import (FermionOperator, jordan_wigner, bravyi_kitaev, get_sparse_operator,
                         get_fermion_operator, hermitian_conjugated)
import os, glob

# compat: newer numpy makes (-1)**int64 a numpy.int64; older openfermion type-checks reject it.
# Must run in EVERY process: joblib/loky workers are fresh processes that don't inherit this patch,
# so encode() (called inside the workers) re-applies it. Guarded -> one-time no-op per process.
def _ofpatch():
    import openfermion.ops.operators.symbolic_operator as _s
    if not isinstance(np.int64(0), _s.COEFFICIENT_TYPES):
        _s.COEFFICIENT_TYPES = tuple(_s.COEFFICIENT_TYPES) + (np.integer, np.floating, np.complexfloating)
_ofpatch()

ENC = ["JW", "BK", "parity"]

# ---- Pauli-propagation engine (gauge_engine/pauli.py, validated vs statevector to 1e-15) ----
def _popcount(x): return int(x).bit_count()
def _commute(xp, zp, xg, zg): return (_popcount(xp & zg) + _popcount(zp & xg)) % 2 == 0
def _pmul(xp, zp, xg, zg):
    x = xp ^ xg; z = zp ^ zg
    sign = -1.0 if (_popcount(zp & xg) & 1) else 1.0
    return x, z, sign
def _conj_gate(terms, xg, zg, theta):
    c = np.cos(2.0 * theta); s = np.sin(2.0 * theta)
    gamma = (1j) ** (_popcount(xg & zg)); out = {}
    for (xp, zp), coeff in terms.items():
        if _commute(xp, zp, xg, zg):
            out[(xp, zp)] = out.get((xp, zp), 0j) + coeff
        else:
            out[(xp, zp)] = out.get((xp, zp), 0j) + coeff * c
            xr, zr, sign = _pmul(xp, zp, xg, zg)
            out[(xr, zr)] = out.get((xr, zr), 0j) + coeff * (-1j) * s * gamma * sign
    return out
def _truncate(terms, w_max, delta):
    if w_max is None and delta <= 0.0: return terms
    out = {}
    for (x, z), c in terms.items():
        if abs(c) < delta: continue
        if w_max is not None and _popcount(x | z) > w_max: continue
        out[(x, z)] = c
    return out
def propagate_circuit(n, gates, obs_terms, w_max=None, delta=0.0):
    terms = dict(obs_terms)
    for (xg, zg, theta) in reversed(gates):
        terms = _conj_gate(terms, xg, zg, theta)
        terms = _truncate(terms, w_max, delta)
    return _truncate(terms, w_max, delta)

# ---- statevector ground truth (gauge_engine/statevec.py) ----
def _pauli_apply(state, x, z, n):
    k = np.arange(2 ** n); sign = np.ones(2 ** n); zz = z
    while zz:
        b = (zz & -zz).bit_length() - 1
        sign = sign * (1 - 2 * ((k >> b) & 1)); zz &= zz - 1
    return (state * sign)[k ^ x]
def apply_rotation(state, x, z, theta, n):
    gamma = (1j) ** (int(x & z).bit_count())
    return np.cos(theta) * state - 1j * np.sin(theta) * (gamma * _pauli_apply(state, x, z, n))

# ---- encodings + helpers ----
def encode(fop, enc, n):
    if enc == "JW": return jordan_wigner(fop)
    if enc == "BK": return bravyi_kitaev(fop, n_qubits=n)
    if enc == "parity":
        _ofpatch()
        return of.binary_code_transform(fop, of.parity_code(n))
    raise ValueError(enc)
def dense(qop, n): return np.asarray(get_sparse_operator(qop, n_qubits=n).todense())
def qop_to_terms(qop):
    out = {}
    for term, coeff in qop.terms.items():
        x = z = 0
        for q, p in term:
            if p in ("X", "Y"): x |= 1 << q
            if p in ("Z", "Y"): z |= 1 << q
        out[(x, z)] = out.get((x, z), 0j) + complex(coeff)
    return {k: v.real for k, v in out.items() if abs(v) > 1e-12}
def max_weight(qop): return max((len(t) for t in qop.terms if t), default=0)

def hermitian_gens(H):
    gens, seen = [], set()
    for term in H.terms:
        if not term: continue
        td = list(hermitian_conjugated(FermionOperator(term)).terms.keys())[0]
        key = tuple(sorted([term, td]))
        if key in seen: continue
        seen.add(key); gens.append(FermionOperator(term) + FermionOperator(td))
    return gens
def circuit(enc, gens, n, angles):
    gates = []
    for layer in angles:
        for theta, g in zip(layer, gens):
            for (x, z), c in qop_to_terms(encode(g, enc, n)).items():
                if (x, z) != (0, 0): gates.append((x, z, theta * c))
    return gates
def hf_index(enc, n, n_elec):
    occ = [1] * n_elec + [0] * (n - n_elec)
    nterms = [qop_to_terms(encode(FermionOperator("%d^ %d" % (i, i)), enc, n)) for i in range(n)]
    for b in range(1 << n):
        if all(abs(sum(c * (1.0 if not (_popcount(z & b) & 1) else -1.0)
                       for (x, z), c in nterms[i].items() if x == 0) - occ[i]) < 1e-6 for i in range(n)):
            return b
    raise RuntimeError("HF not found")
def exp_basis(terms, b):
    return float(sum(np.real(c) * (1.0 if not (_popcount(z & b) & 1) else -1.0)
                     for (x, z), c in terms.items() if x == 0))
def exact_O(n, gates, obs_terms, b):
    psi = np.zeros(1 << n, dtype=complex); psi[b] = 1.0
    for (x, z, th) in gates: psi = apply_rotation(psi, x, z, th, n)
    val = 0j
    for (x, z), a in obs_terms.items():
        val += a * ((1j) ** _popcount(x & z)) * np.vdot(psi, _pauli_apply(psi, x, z, n))
    return val.real
def wstar_hf(n, gates, obs_terms, b, eps=0.05):
    exact = exact_O(n, gates, obs_terms, b)
    for W in range(0, n + 1):
        if abs(exp_basis(propagate_circuit(n, gates, obs_terms, w_max=W), b) - exact) < eps:
            return W, exact
    return n + 1, exact

def two_sre(psi, n):
    d = 1 << n; idx = np.arange(d); pc = np.array([bin(b).count("1") for b in range(d)])
    s2 = s4 = 0.0
    for x in range(d):
        cps = np.conjugate(psi[idx ^ x])
        for z in range(d):
            val = cps @ ((1.0 - 2.0 * (pc[idx & z] & 1)) * psi)
            exp = ((1j) ** (bin(x & z).count("1")) * val).real
            s2 += exp * exp; s4 += exp ** 4
    assert abs(s2 - d) < 1e-6 * d
    return -np.log2(s4 / d)
def dla_dim(gen_mats, cap=800):
    def vec(A): return np.concatenate([A.real.ravel(), A.imag.ravel()]).astype(float)
    basis, Q, queue = [], [], []
    def add(A):
        v = vec(A)
        for q in Q: v = v - (q @ v) * q
        if np.linalg.norm(v) > 1e-7:
            Q.append(v / np.linalg.norm(v)); basis.append(A); return True
        return False
    for M in gen_mats:
        if add(1j * M): queue.append(basis[-1])
    while queue:
        A = queue.pop()
        for B in list(basis):
            C = A @ B - B @ A
            if np.linalg.norm(C) < 1e-9: continue
            if add(C):
                queue.append(basis[-1])
                if len(basis) >= cap: return cap
    return len(basis)

# ---- model families ----
def molecular(N, seed=1):
    rng = np.random.default_rng(seed); H = FermionOperator()
    for p in range(N):
        for q in range(p + 1, N):
            H += rng.normal() * (FermionOperator("%d^ %d" % (p, q)) + FermionOperator("%d^ %d" % (q, p)))
            H += rng.normal() * FermionOperator("%d^ %d %d^ %d" % (p, p, q, q))
    return H
def sparse_gens(n):
    gens = [FermionOperator("%d^ %d" % (i, i + n // 2)) + FermionOperator("%d^ %d" % (i + n // 2, i))
            for i in range(n // 2)]
    gens += [FermionOperator("%d^ %d %d^ %d" % (i, i, i + 1, i + 1)) for i in range(0, n - 1, 2)]
    return gens
def free_fermion_gens(L):
    gens = [FermionOperator("%d^ %d" % (i, i + 1)) + FermionOperator("%d^ %d" % (i + 1, i)) for i in range(L - 1)]
    gens += [FermionOperator("%d^ %d" % (i, i)) for i in range(L)]
    return gens
_DATA = os.path.join(os.path.dirname(of.__file__), "testing", "data")
def load_mol(pattern):
    fs = sorted(f for f in glob.glob(os.path.join(_DATA, pattern + "*.hdf5")) if os.path.basename(f)[0] not in "pt")
    m = of.MolecularData(filename=fs[0][:-5]); m.load()
    return m, get_fermion_operator(m.get_molecular_hamiltonian())

# ---- module-level workers for parallel sweeps (picklable by joblib/loky) ----
def weight_point(args):
    _ofpatch()                       # fresh loky worker process -> ensure compat patch is applied
    N, e = args
    return (N, e, max_weight(encode(molecular(N), e, N)))
def wstar_point(args):
    _ofpatch()                       # fresh loky worker process -> ensure compat patch is applied
    n, e = args
    gens = sparse_gens(n); ne = n // 2
    ang = [list(np.random.default_rng(5).uniform(0.4, 0.85, size=len(gens))) for _ in range(2)]
    b = hf_index(e, n, ne); g = circuit(e, gens, n, ang)
    obs = qop_to_terms(encode(FermionOperator("%d^ %d" % (ne, ne)), e, n))
    return (n, e, wstar_hf(n, g, obs, b)[0])

import os as _os
print("engine loaded. CPU cores available:", _os.cpu_count())

## Step 1 - certify the engine (independent ground truth)

In [ ]:
# ===================== SELF-VALIDATION against independent ground truth =====================
ok = []
# 1 encodings isospectral
m, H = load_mol("H2_sto-3g_singlet_0.7"); n = 4
sp = {e: np.sort(np.linalg.eigvalsh(dense(encode(H, e, n), n)).real) for e in ENC}
ok.append(("isospectral", max(np.max(np.abs(sp["JW"] - sp[e])) for e in ENC) < 1e-9))
# 2 HF energy == molecule stored value
errs = [abs(exp_basis(qop_to_terms(encode(H, e, n)), hf_index(e, n, m.n_electrons)) - m.hf_energy) for e in ENC]
ok.append(("HF energy == mol.hf_energy (%.6f)" % m.hf_energy, max(errs) < 1e-6))
# 3 Heisenberg(w_max=n) == statevector
gens = hermitian_gens(H); ang = [list(np.random.default_rng(1).uniform(0.3, 0.9, size=len(gens)))]
worst = 0.0
for e in ENC:
    b = hf_index(e, n, m.n_electrons); g = circuit(e, gens, n, ang)
    obs = qop_to_terms(encode(FermionOperator("1^ 1"), e, n))
    worst = max(worst, abs(exp_basis(propagate_circuit(n, g, obs, w_max=n), b) - exact_O(n, g, obs, b)))
ok.append(("engine == statevector", worst < 1e-7))
# 4 DLA == u(L) = L^2
ok.append(("DLA == L^2", all(dla_dim([dense(encode(gn, e, 4), 4) for gn in free_fermion_gens(4)]) == 16 for e in ENC)))
# 5 magic SRE analytic
t = np.array([1.0, np.exp(1j * np.pi / 4)]) / np.sqrt(2)
ok.append(("SRE T-state == log2(4/3)", abs(two_sre(t, 1) - np.log2(4 / 3)) < 1e-6 and abs(two_sre(np.array([1.0, 0.0]), 1)) < 1e-9))
# 6 weight covariant
g6 = FermionOperator("0^ 7") + FermionOperator("7^ 0")
ok.append(("weight covariant JW>BK", max_weight(encode(g6, "JW", 8)) == 8 and max_weight(encode(g6, "BK", 8)) < 8))
for name, p in ok: print("[%s] %s" % ("PASS" if p else "FAIL", name))
assert all(p for _, p in ok), "VALIDATION FAILED -- do not trust the study below"
print("\n>>> all %d checks PASSED: engine is correct on this machine.\n" % len(ok))

## Step 2 - LiH/STO-3G UCCSD: operating-point sweep and the verdict flip test

In [ ]:
# ===================== LiH/STO-3G (n=12): an honest molecular W* flip test =====================
import numpy as np, time
from joblib import Parallel, delayed

def uccsd_gens(n, ne):
    """Standard UCCSD Hermitian generators: singles + doubles (occupied <-> virtual)."""
    occ, virt = list(range(ne)), list(range(ne, n)); g = []
    for i in occ:
        for a in virt:
            g.append(FermionOperator("%d^ %d" % (a, i)) + FermionOperator("%d^ %d" % (i, a)))
    for ii in range(len(occ)):
        for jj in range(ii + 1, len(occ)):
            for aa in range(len(virt)):
                for bb in range(aa + 1, len(virt)):
                    i, j, a, b = occ[ii], occ[jj], virt[aa], virt[bb]
                    g.append(FermionOperator("%d^ %d^ %d %d" % (a, b, j, i))
                             + FermionOperator("%d^ %d^ %d %d" % (i, j, b, a)))
    return g

m, H = load_mol("H1-Li1_sto-3g_singlet_1.45")
n, ne = m.n_qubits, m.n_electrons
thr = 2 * np.log2(n); Wchk = int(np.floor(thr)); obs_mode = ne - 1; eps = 0.05
gens = uccsd_gens(n, ne)
print("LiH/STO-3G: n=%d qubits, %d electrons, %d UCCSD generators, threshold=%.2f (verdict at W=%d), eps=%.2f"
      % (n, ne, len(gens), thr, Wchk, eps), flush=True)

# --- ground-truth check: encoded Hartree-Fock energy matches the stored molecular value ---
for e in ENC:
    hf = exp_basis(qop_to_terms(encode(H, e, n)), hf_index(e, n, ne))
    print("  [HF check %-7s] <HF|H|HF>=%.6f  stored=%.6f  |err|=%.1e" % (e, hf, m.hf_energy, abs(hf - m.hf_energy)), flush=True)

def verdict_point(args):
    _ofpatch(); enc, amp, seed, layers = args
    g = uccsd_gens(n, ne)
    ang = [list(np.random.default_rng(seed + L).uniform(0.15, amp, size=len(g))) for L in range(layers)]
    b = hf_index(enc, n, ne); gates = circuit(enc, g, n, ang)
    obs = qop_to_terms(encode(FermionOperator("%d^ %d" % (obs_mode, obs_mode)), enc, n))
    ex = exact_O(n, gates, obs, b)
    err = abs(exp_basis(propagate_circuit(n, gates, obs, w_max=Wchk), b) - ex)
    return (enc, amp, seed, layers, ex, err)

# --- the sweep: operating point x seed, all three encodings, verdict at the fixed threshold ---
AMPS = [0.5, 0.7, 0.9, 1.1, 1.3]
SEEDS = [100, 200]
LAYERS = 1
tasks = [(e, a, s, LAYERS) for a in AMPS for s in SEEDS for e in ENC]
t0 = time.time()
res = Parallel(n_jobs=-1)(delayed(verdict_point)(t) for t in tasks)
print("\nsweep done in %.0f s\n" % (time.time() - t0), flush=True)
R = {(e, a, s): (ex, err) for (e, a, s, L, ex, err) in res}

print("amp   seed | <O> spread | JW err  verdict | BK err  verdict | par err verdict | FLIP?")
nflip = 0
for a in AMPS:
    for s in SEEDS:
        os_ = [R[(e, a, s)][0] for e in ENC]; spread = max(os_) - min(os_)
        v = {e: ("HARD" if R[(e, a, s)][1] > eps else "sim ") for e in ENC}
        flip = (v["JW"] == "HARD" and v["BK"] == "sim " and spread < 1e-3)
        nflip += int(flip)
        print("%-5.2f %-4d | %.1e   | %.4f %s | %.4f %s | %.4f %s | %s"
              % (a, s, spread, R[("JW", a, s)][1], v["JW"], R[("BK", a, s)][1], v["BK"],
                 R[("parity", a, s)][1], v["parity"], "*** FLIP ***" if flip else ""))
print("\nflip points: %d / %d (JW HARD, BK simulable, <O> invariant). Threshold fixed at 2 log2 n." % (nflip, len(AMPS) * len(SEEDS)))

# --- if a flip is robust, report the actual integer W* at a representative point ---
def wstar_point(args):
    _ofpatch(); enc, amp, seed, layers = args
    g = uccsd_gens(n, ne)
    ang = [list(np.random.default_rng(seed + L).uniform(0.15, amp, size=len(g))) for L in range(layers)]
    b = hf_index(enc, n, ne); gates = circuit(enc, g, n, ang)
    obs = qop_to_terms(encode(FermionOperator("%d^ %d" % (obs_mode, obs_mode)), enc, n))
    ex = exact_O(n, gates, obs, b)
    Wcap = min(n, Wchk + 3)                       # bound the expensive high-W propagation for HARD encodings
    for W in range(0, Wcap + 1):
        if abs(exp_basis(propagate_circuit(n, gates, obs, w_max=W), b) - ex) < eps:
            return (enc, W, ex)
    return (enc, Wcap + 1, ex)                     # ">Wcap" : still HARD past the cap

if nflip > 0:
    bestamp = 0.9 if 0.9 in AMPS else AMPS[len(AMPS) // 2]
    print("\nfull integer W* at amp=%.2f, seed=%d (representative):" % (bestamp, SEEDS[0]))
    ws = Parallel(n_jobs=-1)(delayed(wstar_point)((e, bestamp, SEEDS[0], LAYERS)) for e in ENC)
    for (e, W, ex) in ws:
        print("  %-7s W*=%-4s (%s)  <O>=%+.4f" % (e, ("%d" % W if W <= n else ">%d" % n), "HARD" if W > thr else "simulable", ex))
print("\n=> A flip here is a real molecular instance where the simulability verdict is encoding-relative;")
print("   no flip => the H2/LiH molecular data remain an invariance check, as the paper already states.")

## Reading the result
- **If FLIP points appear** (JW HARD, BK simulable, `<O>` spread < 1e-3): a real molecule exhibits the
  encoding-relative verdict -- promote the chemistry from a check to a demonstration.
- **If no flip**: the molecular instances stay an invariance check, exactly as the paper states; the
  verdict-flip demonstration remains the synthetic structured family. Either outcome is reported as-is.
